In [34]:
# data prep
import pandas as pd
from pathlib import Path

path_to_data = "/Users/diasirishev/Desktop/CodeBase/transformer-workshop/docs/holy.txt"

text = Path(path_to_data).read_text(encoding="utf-8")

split_1 = int(len(text)*0.9)
split_2 = int(len(text)*0.95)

train_text = text[:split_1]
val_text = text[split_1:split_2]
test_text = text[split_2:]


In [35]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)
tokenizer.decoder = decoders.ByteLevel()


trainer = trainers.BpeTrainer(
    vocab_size=2048,
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

tokenizer.train_from_iterator(
    train_text.splitlines(keepends=True),
    trainer=trainer,
)


sample = "The evening was quiet."
encoded = tokenizer.encode(sample)

print(encoded.tokens)  # Token pieces
print(encoded.ids)     # Integer IDs
print(tokenizer.decode(encoded.ids))

assert tokenizer.decode(encoded.ids) == sample


tokenizer.save("book_tokenizer.json")

train_ids = tokenizer.encode(train_text).ids
val_ids = tokenizer.encode(val_text).ids

vocab_size = tokenizer.get_vocab_size()
print("Vocabulary:", vocab_size)
print("Training tokens:", len(train_ids))
print("Validation tokens:", len(val_ids))




['The', 'Ġeven', 'ing', 'Ġwas', 'Ġqu', 'i', 'et', '.']
[334, 902, 294, 470, 1221, 72, 622, 13]
The evening was quiet.
Vocabulary: 2048
Training tokens: 211614
Validation tokens: 12102


In [ ]:
from classic_transformer import Transformer_Block, Embedding_Block, PositionalEncoding_Block
from torch import nn
import torch

D = 256
num_heads = 4
num_layers = 4
hidden_dim = 4 * D
max_length = 128



In [37]:
class NanoGPT(nn.Module):
    def __init__(self, D, num_heads, num_layers, hidden_dim, max_length, vocab_size):
        super().__init__()
        self.blocks = nn.ModuleList([
            Transformer_Block(D, num_heads, hidden_dim)
            for _ in range(num_layers)
        ])
        self.embedding = Embedding_Block(vocab_size, D)
        self.positional_encoding = PositionalEncoding_Block(D, max_length)
        self.layer_norm = nn.LayerNorm(D)
        self.vocab_logit = nn.Linear(D, vocab_size)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        X = self.embedding(X)
        X = self.positional_encoding(X)
        for block in self.blocks:
            X = block(X)
        X = self.layer_norm(X)
        X = self.vocab_logit(X)
        return X

        

In [38]:
# Keep the token stream on CPU; move only the sampled batch to MPS.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
train_data = torch.tensor(train_ids, dtype=torch.long)
batch_size = 8
sequence_length = 128
assert sequence_length <= max_length


def get_batch(data, batch_size, sequence_length, device):
    """Sample consecutive chunks; return shifted tensors shaped (B, I)."""
    if data.ndim != 1 or len(data) <= sequence_length:
        raise ValueError("Need a 1D token stream with at least sequence_length + 1 tokens")
    if batch_size <= 0 or sequence_length <= 0:
        raise ValueError("batch_size and sequence_length must be positive")

    # randint's upper bound is exclusive; leave room for the final target.
    starts = torch.randint(0, len(data) - sequence_length, (batch_size,))
    inputs = torch.stack([data[s:s + sequence_length] for s in starts.tolist()])
    targets = torch.stack([data[s + 1:s + sequence_length + 1] for s in starts.tolist()])
    return inputs.to(device), targets.to(device)


# Rerunning this cell creates a NEW model, starting training from scratch.
model = NanoGPT(
    D=D,
    num_heads=num_heads,
    num_layers=num_layers,
    hidden_dim=hidden_dim,
    max_length=max_length,
    vocab_size=vocab_size,
)
# Small initial projection weights help avoid saturated attention scores.
# LayerNorm scales stay at their default of 1; biases start at zero.
with torch.no_grad():
    for name, parameter in model.named_parameters():
        if name.endswith(".bias") or any(f".{b}." in name for b in ("Q_b", "K_b", "V_b")):
            parameter.zero_()
        elif parameter.ndim >= 2:
            parameter.normal_(mean=0.0, std=0.02)
model = model.to(device)
print("Training device:", device)


Training device: mps


In [39]:
# Inspect one random batch before training.
inputs, targets = get_batch(train_data, batch_size, sequence_length, device)
assert torch.equal(inputs[:, 1:], targets[:, :-1])
with torch.no_grad():
    logits = model(inputs)
    initial_loss = nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size), targets.reshape(-1)
    )
print("Inputs / targets:", inputs.shape, targets.shape)  # (8, 128)
print("Logits:", logits.shape)  # (8, 128, vocab_size)
print("Initial loss:", initial_loss.item())


Inputs / targets: torch.Size([8, 128]) torch.Size([8, 128])
Logits: torch.Size([8, 128, 2048])
Initial loss: 7.634530067443848


### Multi-batch training and checkpoints

Default: D=256, 4 layers, 4 heads, context 128, batch size 8 (~4.2M parameters at vocabulary 2048). Restarting the setup creates a new model.

Every 100 steps and at the end, save `latest.pt` with model/optimizer states, configuration, tokenizer JSON, step and loss history. Each fresh run gets its own directory under the repository's ignored `checkpoints/` directory. Saves replace the latest checkpoint atomically instead of accumulating files.

To resume, set `resume_from` in the training cell to a saved `latest.pt`. This restores the architecture, weights, optimizer and tokenizer; `total_steps` is the target cumulative step count. Random batch sampling is restarted, so this is not a bit-for-bit replay. Validation is still a separate next exercise.


In [ ]:
from datetime import datetime
from pathlib import Path
from tokenizers import Tokenizer

resume_from = None  # Example: Path("../checkpoints/nanogpt-.../latest.pt")
total_steps = 10000  # Cumulative target when resuming.
log_every = 25
save_every = 100

config = dict(D=D, num_heads=num_heads, num_layers=num_layers,
              hidden_dim=hidden_dim, max_length=max_length, vocab_size=vocab_size)
start_step = 0
loss_history = []
if resume_from is not None:
    checkpoint = torch.load(resume_from, map_location="cpu", weights_only=True)
    config = checkpoint["config"]
    tokenizer = Tokenizer.from_str(checkpoint["tokenizer_json"])
    vocab_size = config["vocab_size"]
    max_length = config["max_length"]
    sequence_length = checkpoint["sequence_length"]
    batch_size = checkpoint["batch_size"]
    train_ids = tokenizer.encode(train_text).ids
    train_data = torch.tensor(train_ids, dtype=torch.long)
    model = NanoGPT(**config).to(device)
    model.load_state_dict(checkpoint["model"])
    start_step = checkpoint["step"]
    loss_history = checkpoint["loss_history"]
    checkpoint_dir = Path(resume_from).resolve().parent
else:
    # Works from the repository root or from attention_mechanism/.
    root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "attention_mechanism/classic_transformer.py").exists())
    checkpoint_dir = root / "checkpoints" / datetime.now().strftime("nanogpt-%Y%m%d-%H%M%S-%f")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
if resume_from is not None:
    opt.load_state_dict(checkpoint["optimizer"])
    del checkpoint
loss_fn = nn.CrossEntropyLoss()


def to_cpu(value):
    # Include AdamW's moving averages, not only the model weights.
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {k: to_cpu(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_cpu(v) for v in value]
    if isinstance(value, tuple):
        return tuple(to_cpu(v) for v in value)
    return value


def save_checkpoint(completed_steps):
    payload = dict(model=to_cpu(model.state_dict()), optimizer=to_cpu(opt.state_dict()),
                   config=config, step=completed_steps, loss_history=loss_history,
                   tokenizer_json=tokenizer.to_str(), sequence_length=sequence_length,
                   batch_size=batch_size)
    temporary = checkpoint_dir / "latest.tmp"
    torch.save(payload, temporary)
    temporary.replace(checkpoint_dir / "latest.pt")
    print(f"Saved step {completed_steps}: {checkpoint_dir / 'latest.pt'}")


print("Parameters:", sum(p.numel() for p in model.parameters()))
running_loss = 0.0
window_steps = 0
completed_steps = start_step
model.train()
try:
    for step in range(start_step, total_steps):
        inputs, targets = get_batch(train_data, batch_size, sequence_length, device)
        opt.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = loss_fn(logits.reshape(-1, vocab_size), targets.reshape(-1))
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        completed_steps = step + 1
        running_loss += loss.item()
        window_steps += 1
        if completed_steps % log_every == 0 or completed_steps == total_steps:
            mean_loss = running_loss / window_steps
            loss_history.append((completed_steps, mean_loss))
            print(f"Step {completed_steps}/{total_steps}: mean training loss = {mean_loss:.4f}")
            running_loss = 0.0
            window_steps = 0
        if completed_steps % save_every == 0:
            save_checkpoint(completed_steps)
except KeyboardInterrupt:
    print("Interrupted; saving current state.")
# Save after normal completion or a keyboard interruption.
save_checkpoint(completed_steps)


### Inference

In [ ]:
model.eval()


FIRST_PROMPT = "Is there a god?"
first_tokens = tokenizer.encode(FIRST_PROMPT).ids
first_tokens = torch.tensor(
    [first_tokens], dtype=torch.long, device=device
)

print(FIRST_PROMPT)
TOTAL_GENERATION_LEN = 500
with torch.no_grad():
    for _ in range(TOTAL_GENERATION_LEN):
        logits = model(first_tokens[:, -max_length:])
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        decoded_token = tokenizer.decode(next_token[0].tolist())
        print(decoded_token)
        first_tokens = torch.cat((first_tokens, next_token), dim=1)

AttributeError: 'tokenizers.Encoding' object has no attribute 'ndim'

In [ ]:

first_tokens

Encoding(num_tokens=5, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])